Purpose: Filter genes for input to XGBoost.<br>
Author: Anna Pardo<br>
Date initiated: Feb. 17, 2026

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold

In [2]:
# load input data
indata = pd.read_csv("./paired_TPM_physiology.txt",sep="\t",header="infer")
indata.head()

/tmp/ipykernel_19819/404855715.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  indata = pd.read_csv("./paired_TPM_physiology.txt",sep="\t",header="infer")


,sample_name,genotype,treat,ZT,photo,cond,Yucal.01G000100.v2.1,Yucal.01G000200.v2.1,Yucal.01G000300.v2.1,Yucal.01G000400.v2.1,...,YufilH1095122m.g,YufilH1095123m.g,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095146m.g,YufilH1095147m.g
0,Y111,18,D,1.0,2.627596,0.026132,48.380647,6.981615,0.00000,19.211534,...,0.000000,5.500249,2.171056,0.671994,8.838875,0.372084,2.805992,4.407968,0.0,0.000000
1,Y123,18,D,1.0,2.679172,0.021603,52.585871,5.331016,0.57518,12.811805,...,0.101045,11.410506,2.456701,0.464693,10.125277,0.327475,1.058391,4.115716,0.0,0.000000
2,Y120,18,D,3.0,1.211527,0.007856,41.192373,5.821853,0.00000,16.055452,...,0.000000,5.037032,2.061853,0.896509,10.437337,0.302887,1.945765,5.470556,0.0,0.212331
3,Y125,18,D,3.0,1.289725,0.009783,49.458436,6.239587,0.00000,20.374043,...,0.000000,6.796193,3.347733,0.445386,9.763749,0.422761,2.479682,5.295689,0.0,0.000000
4,Y129,18,D,3.0,1.549238,0.010471,45.312890,7.263022,0.00000,19.973103,...,0.116594,7.899789,1.574850,0.584944,9.166182,0.539808,1.944959,7.736828,0.0,0.000000


In [29]:
# filter to only drought samples
droughtonly = indata[indata["treat"]=="D"]
len(droughtonly.index)

211

In [32]:
# ensure all genotypes are of type=str
droughtonly["genotype"] = droughtonly["genotype"].astype(str)
droughtonly["genotype"].unique()

/tmp/ipykernel_19819/2947304792.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  droughtonly["genotype"] = droughtonly["genotype"].astype(str)


array(['18', '1AB', '2AB', '15', 'Eudy', '13', '19', '45', 'G', '36',
       '48', '56', '52', '37', '43'], dtype=object)

In [33]:
# drop genotypes that have only one sample each (37, 43, 48)
droughtonly = droughtonly[~droughtonly["genotype"].isin(["37","43","48"])]

In [34]:
len(droughtonly.index)

208

In [35]:
len(droughtonly.columns)-6

85962

Given that I have many more genes (features) than observations, the curse of dimensionality is very much an issue here. Some non-arbitrary gene sets I have as options to take care of that:<br>
- maSigPro clusters peaking at different ZTs (for each genotype...would need to handle this diversity somehow)
- HEB genes (biased to Yf or biased to Ya)
- recently generated: expression partitions from HybridExpress (again, for each genotype)
- CAM genes

Also, I should remove zero-variance features.

In [8]:
# function for filtering out zero-variance features
# define a function from an answer in https://stackoverflow.com/questions/39812885/retain-feature-names-after-scikit-feature-selection
def variance_threshold_selector(data):
    selector = VarianceThreshold()
    selector.fit(data)
    return data[data.columns[selector.get_support(indices=True)]]

In [10]:
droughtonly.head()

,sample_name,genotype,treat,ZT,photo,cond,Yucal.01G000100.v2.1,Yucal.01G000200.v2.1,Yucal.01G000300.v2.1,Yucal.01G000400.v2.1,...,YufilH1095122m.g,YufilH1095123m.g,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095146m.g,YufilH1095147m.g
0,Y111,18,D,1.0,2.627596,0.026132,48.380647,6.981615,0.00000,19.211534,...,0.000000,5.500249,2.171056,0.671994,8.838875,0.372084,2.805992,4.407968,0.0,0.000000
1,Y123,18,D,1.0,2.679172,0.021603,52.585871,5.331016,0.57518,12.811805,...,0.101045,11.410506,2.456701,0.464693,10.125277,0.327475,1.058391,4.115716,0.0,0.000000
2,Y120,18,D,3.0,1.211527,0.007856,41.192373,5.821853,0.00000,16.055452,...,0.000000,5.037032,2.061853,0.896509,10.437337,0.302887,1.945765,5.470556,0.0,0.212331
3,Y125,18,D,3.0,1.289725,0.009783,49.458436,6.239587,0.00000,20.374043,...,0.000000,6.796193,3.347733,0.445386,9.763749,0.422761,2.479682,5.295689,0.0,0.000000
4,Y129,18,D,3.0,1.549238,0.010471,45.312890,7.263022,0.00000,19.973103,...,0.116594,7.899789,1.574850,0.584944,9.166182,0.539808,1.944959,7.736828,0.0,0.000000


In [36]:
# log transform & drop zero-variance genes
## start by setting metadata & response vars as the index
droughtonly.set_index(["sample_name","genotype","treat","ZT","photo","cond"],inplace=True)

In [37]:
droughtonly.head()

,,,,,,Yucal.01G000100.v2.1,Yucal.01G000200.v2.1,Yucal.01G000300.v2.1,Yucal.01G000400.v2.1,Yucal.01G000500.v2.1,Yucal.01G000600.v2.1,Yucal.01G000700.v2.1,Yucal.01G000800.v2.1,Yucal.01G000900.v2.1,Yucal.01G001000.v2.1,...,YufilH1095122m.g,YufilH1095123m.g,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095146m.g,YufilH1095147m.g
sample_name,genotype,treat,ZT,photo,cond,,,,,,,,,,,,,,,,,,,,,
Y111,18,D,1.0,2.627596,0.026132,48.380647,6.981615,0.00000,19.211534,3.029870,11.992049,8.627981,0.000000,5.910205,38.140510,...,0.000000,5.500249,2.171056,0.671994,8.838875,0.372084,2.805992,4.407968,0.0,0.000000
Y123,18,D,1.0,2.679172,0.021603,52.585871,5.331016,0.57518,12.811805,2.532772,8.899058,4.117794,0.000000,5.500478,39.477648,...,0.101045,11.410506,2.456701,0.464693,10.125277,0.327475,1.058391,4.115716,0.0,0.000000
Y120,18,D,3.0,1.211527,0.007856,41.192373,5.821853,0.00000,16.055452,2.633051,6.684763,6.450086,0.000000,6.745969,24.998323,...,0.000000,5.037032,2.061853,0.896509,10.437337,0.302887,1.945765,5.470556,0.0,0.212331
Y125,18,D,3.0,1.289725,0.009783,49.458436,6.239587,0.00000,20.374043,2.930805,5.775961,7.502370,0.000000,7.116615,19.912820,...,0.000000,6.796193,3.347733,0.445386,9.763749,0.422761,2.479682,5.295689,0.0,0.000000
Y129,18,D,3.0,1.549238,0.010471,45.312890,7.263022,0.00000,19.973103,3.207628,7.091446,6.590688,0.221229,7.325466,20.691437,...,0.116594,7.899789,1.574850,0.584944,9.166182,0.539808,1.944959,7.736828,0.0,0.000000


In [38]:
vttpm = variance_threshold_selector(droughtonly)

In [39]:
print("Number of zero-variance genes removed:",len(droughtonly.columns)-len(vttpm.columns))

Number of zero-variance genes removed: 3925


In [40]:
vttpm_log = vttpm.apply(lambda x: np.log2(x+1))

In [41]:
vttpm_log.reset_index(inplace=True)
vttpm_log.head()

,sample_name,genotype,treat,ZT,photo,cond,Yucal.01G000100.v2.1,Yucal.01G000200.v2.1,Yucal.01G000300.v2.1,Yucal.01G000400.v2.1,...,YufilH1095121m.g,YufilH1095122m.g,YufilH1095123m.g,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095147m.g
0,Y111,18,D,1.0,2.627596,0.026132,5.625874,2.996681,0.000000,4.337107,...,1.655155,0.000000,2.700495,1.664964,0.741569,3.298493,0.456369,1.928273,2.435087,0.000000
1,Y123,18,D,1.0,2.679172,0.021603,5.743781,2.662437,0.655517,3.787830,...,2.617073,0.138874,3.633490,1.789396,0.550599,3.475769,0.408684,1.041517,2.354936,0.000000
2,Y120,18,D,3.0,1.211527,0.007856,5.398910,2.770164,0.000000,4.092161,...,2.574780,0.000000,2.593839,1.614405,0.923346,3.515679,0.381712,1.558642,2.693890,0.277784
3,Y125,18,D,3.0,1.289725,0.009783,5.657024,2.855907,0.000000,4.417788,...,2.480106,0.000000,2.962770,2.120263,0.531454,3.428109,0.508694,1.798956,2.654364,0.000000
4,Y129,18,D,3.0,1.549238,0.010471,5.533342,3.046669,0.000000,4.390468,...,2.221651,0.159104,3.153771,1.364488,0.664432,3.345706,0.622750,1.558247,3.127110,0.000000


In [42]:
# save vttpm_log
vttpm_log.to_csv("./allfeatures_paired_logTPM_phys_forxgboost.txt",sep="\t",header=True,index=False)

In [48]:
# set up a function to subset to a specified set of genes
def subset_genes(genelist,data=vttpm_log):
    data = data.set_index(["sample_name","genotype","treat","ZT","photo","cond"])
    subdata = data[data.columns.intersection(genelist)]
    return subdata.reset_index()

In [21]:
# load CAM genes
cam = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/degs_downstream/camgenes_Ya_Yf_orthology_synteny.txt",
                 sep="\t",header="infer")
cam.head()

,GeneID,Orthogroup,Pathway,gene_name,gene_abbr,gene_abbr_unique,subgenome
0,Yucal.01G165600.v2.1,OG0001578,CAM-dark,beta-carbonic anhydrase,bCA1234,Ya_bCA1234_1,Ya
1,Yucal.02G112700.v2.1,OG0001578,CAM-dark,beta-carbonic anhydrase,bCA1234,Ya_bCA1234_2,Ya
2,Yucal.04G001000.v2.1,OG0004406,CAM-dark,beta-carbonic anhydrase,bCA5,Ya_bCA5_1,Ya
3,Yucal.07G000800.v2.1,OG0004406,CAM-dark,beta-carbonic anhydrase,bCA5,Ya_bCA5_2,Ya
4,Yucal.03G120900.v2.1,OG0002899,CAM-dark,NAD-dependent malate dehydrogenase (chloroplas...,NAD-MDH-cp,Ya_NAD-MDH-cp_1,Ya


In [49]:
camfeatures = subset_genes(list(cam["GeneID"].unique()))
len(camfeatures.columns)-6

70

In [45]:
camfeatures.head()

,sample_name,genotype,treat,ZT,photo,cond,Yucal.01G077800.v2.1,Yucal.01G137500.v2.1,Yucal.01G165600.v2.1,Yucal.02G112700.v2.1,...,YufilH1069008m.g,YufilH1069011m.g,YufilH1069012m.g,YufilH1072272m.g,YufilH1072277m.g,YufilH1075687m.g,YufilH1082305m.g,YufilH1082501m.g,YufilH1086337m.g,YufilH1086489m.g
0,Y111,18,D,1.0,2.627596,0.026132,5.207348,1.985238,9.233768,8.540209,...,5.908476,0.0,0.0,0.0,4.795117,0.000000,3.197020,1.091027,0.293100,0.860700
1,Y123,18,D,1.0,2.679172,0.021603,4.552644,1.349067,9.081378,7.905170,...,5.902871,0.0,0.0,0.0,4.969240,0.341290,2.639020,1.089835,0.191078,0.387094
2,Y120,18,D,3.0,1.211527,0.007856,4.965965,2.080715,10.231111,10.600430,...,5.799164,0.0,0.0,0.0,4.598032,0.229951,2.514945,2.464440,0.315569,0.244272
3,Y125,18,D,3.0,1.289725,0.009783,5.374245,1.907274,9.716797,9.205871,...,5.965906,0.0,0.0,0.0,4.768673,0.311715,2.733348,2.796774,0.468769,0.300723
4,Y129,18,D,3.0,1.549238,0.010471,5.471138,2.077216,9.946447,9.425045,...,5.796786,0.0,0.0,0.0,4.731755,0.167593,2.746976,2.377347,0.134937,0.395782


In [50]:
len(camfeatures.index)

208

In [51]:
camfeatures.to_csv("./CAMgenes_logTPM_phys_forxgboost.txt",sep="\t",header=True,index=False)